# 02 Preprocessing
Crops the occipital lobe around V1-V3, builds tissue maps and surfaces, fits the reference fODF, and prepares network inputs.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
subjects = PL.make_subject_lists(cfg)
main = [PL.session(s) for s in dict.fromkeys(subjects["test"] + subjects["train"] + subjects["val"] + subjects["retest"])]
extra = [PL.session(s, "7T") for s in subjects["t7"]] + [PL.session(s, "retest") for s in subjects["retest"]]
from tqdm.auto import tqdm
with Timer(cfg, "02_crop"):
    for ses in tqdm(main + extra): PL.prepare(cfg, ses)

In [ ]:
# Reference fODF (multi-shell CSD on the full 1.25 mm data) and network inputs
fit_dev = device if device.type == "cuda" else "cpu"
with Timer(cfg, "02_reference_fodf"):
    for ses in tqdm(main + [s for s in extra if s["kind"] == "retest"]): PL.fodf(cfg, ses, "REF", "msmt_csd", fit_dev)
with Timer(cfg, "02_network_inputs"):
    for s in tqdm(subjects["train"] + subjects["val"]):
        for p in cfg.train_protocols:
            PL.net_input(cfg, PL.session(s), p)
            if "fodrefine" in cfg.variants: PL.net_input(cfg, PL.session(s), p, source="fodf")   # classical fODF as input

In [ ]:
# Quality check on one subject: tissue fractions, seeds, fODF energy
a = PL.load_anat(cfg, main[0]); f = PL.fodf(cfg, main[0], "REF", "msmt_csd"); z = int(a["shape"][2] // 2)
v = np.flatnonzero(a["area"] > 0); near = np.abs(a["verts_vox"][v, 2] - z) < 1
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(a["pve"][0][:, :, z].T, origin="lower", cmap="gray"); ax[0].set_title("white-matter fraction")
ax[0].scatter(a["verts_vox"][v[near], 0], a["verts_vox"][v[near], 1], c=a["area"][v[near]], s=6, cmap="viridis")
ax[1].imshow(a["sdist"][:, :, z].T, origin="lower", cmap="coolwarm", vmin=-5, vmax=5); ax[1].set_title("distance to white surface (mm)")
ax[2].imshow(np.linalg.norm(f["sh"][1:, :, :, z], axis=0).T, origin="lower", cmap="magma"); ax[2].set_title("reference fODF anisotropic energy")
[x.axis("off") for x in ax]; fig.savefig(FIG / "qc_preprocessing.png", dpi=150, bbox_inches="tight")
print("subject", a["sid"], "| crop", a["shape"], "| V1-V3 vertices", len(v), "| with retinotopy:", "prf_ang" in a)